# 08b - MediaPipe Preprocessing: WLASL100

Mismo proceso que notebook 08, pero para **WLASL100** (100 palabras, ~1000 videos).

Esto nos dará ~3x más datos para entrenar y debería mejorar significativamente la generalizacion.

In [1]:
%pip install -q mediapipe opencv-python numpy tqdm


[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# ==========================================
# 1. IMPORTS Y CONFIGURACION
# ==========================================
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import urllib.request
import os
import cv2
import numpy as np
import json
from pathlib import Path
from tqdm import tqdm
from typing import List, Tuple, Dict, Optional
import warnings
warnings.filterwarnings('ignore')

print(f"MediaPipe version: {mp.__version__}")

MediaPipe version: 0.10.32


In [3]:
# ==========================================
# 2. CONFIGURACION DE RUTAS
# ==========================================

# WLASL100 (en vez de WLASL30)
WLASL100_ROOT = Path("/home/fransando/Dev/tfg-sign-language-recognition/06_datasets/wlasl/processed/wlasl100")
RAW_VIDEOS_DIR = Path("/home/fransando/Dev/tfg-sign-language-recognition/06_datasets/wlasl/raw/kaggle_wlasl/videos")

# Output para landmarks de WLASL100
OUTPUT_ROOT = Path("/home/fransando/Dev/tfg-sign-language-recognition/06_datasets/wlasl/landmarks/wlasl100")

# Configuracion
MAX_FRAMES = 64
MIN_DETECTION_RATE = 0.3

print(f"WLASL100 root: {WLASL100_ROOT}")
print(f"  Existe: {WLASL100_ROOT.exists()}")
print(f"\nOutput: {OUTPUT_ROOT}")

WLASL100 root: /home/fransando/Dev/tfg-sign-language-recognition/06_datasets/wlasl/processed/wlasl100
  Existe: True

Output: /home/fransando/Dev/tfg-sign-language-recognition/06_datasets/wlasl/landmarks/wlasl100


In [4]:
# ==========================================
# 3. CARGAR METADATA
# ==========================================

with open(WLASL100_ROOT / 'metadata.json', 'r') as f:
    metadata = json.load(f)

print("=== WLASL100 Dataset ===")
print(f"Total videos: {metadata['statistics']['total_sequences']}")
print(f"  - Train: {metadata['statistics']['train']}")
print(f"  - Val: {metadata['statistics']['val']}")
print(f"  - Test: {metadata['statistics']['test']}")
print(f"\nPalabras: {metadata['statistics']['total_words']}")
print(f"Glosses ejemplo: {metadata['glosses'][:10]}...")

=== WLASL100 Dataset ===
Total videos: 1013
  - Train: 748
  - Val: 165
  - Test: 100

Palabras: 100
Glosses ejemplo: ['accident', 'africa', 'all', 'apple', 'basketball', 'bed', 'before', 'bird', 'birthday', 'black']...


In [5]:
# ==========================================
# 4. INICIALIZAR MEDIAPIPE
# ==========================================

model_path = "hand_landmarker.task"
if not os.path.exists(model_path):
    print("Descargando modelo...")
    url = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
    urllib.request.urlretrieve(url, model_path)

base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=2,
    min_hand_detection_confidence=0.3,
    min_hand_presence_confidence=0.3,
    min_tracking_confidence=0.3
)
detector = vision.HandLandmarker.create_from_options(options)
print("MediaPipe inicializado.")

MediaPipe inicializado.


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1770125284.140930   69896 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1770125284.157217   69911 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [6]:
# ==========================================
# 5. FUNCIONES (Copiadas de notebook 08)
# ==========================================

def load_video_frames(video_path: Path, max_frames: int = 64) -> List[np.ndarray]:
    """Carga frames de un video MP4."""
    cap = cv2.VideoCapture(str(video_path))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    if total_frames == 0:
        cap.release()
        return []
    
    if total_frames > max_frames:
        indices = set(np.linspace(0, total_frames-1, max_frames, dtype=int))
    else:
        indices = set(range(total_frames))
    
    frames = []
    frame_idx = 0
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx in indices:
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame_rgb)
        frame_idx += 1
    
    cap.release()
    return frames


def extract_landmarks_from_frame(frame: np.ndarray, hands_detector) -> Optional[np.ndarray]:
    """Extrae landmarks de un frame."""
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)
    result = hands_detector.detect(mp_image)
    
    if result.hand_landmarks:
        landmarks = result.hand_landmarks[0]
        coords = np.array([[lm.x, lm.y, lm.z] for lm in landmarks])
        return coords
    return None


def interpolate_missing_frames(landmarks_list: List[Optional[np.ndarray]]) -> np.ndarray:
    """Interpola frames sin deteccion."""
    n_frames = len(landmarks_list)
    result = np.zeros((n_frames, 21, 3))
    
    valid_indices = [i for i, lm in enumerate(landmarks_list) if lm is not None]
    
    if len(valid_indices) == 0:
        return result
    
    for i in valid_indices:
        result[i] = landmarks_list[i]
    
    for i in range(n_frames):
        if landmarks_list[i] is not None:
            continue
        
        prev_valid = None
        for j in range(i-1, -1, -1):
            if landmarks_list[j] is not None:
                prev_valid = j
                break
        
        next_valid = None
        for j in range(i+1, n_frames):
            if landmarks_list[j] is not None:
                next_valid = j
                break
        
        if prev_valid is not None and next_valid is not None:
            alpha = (i - prev_valid) / (next_valid - prev_valid)
            result[i] = (1 - alpha) * landmarks_list[prev_valid] + alpha * landmarks_list[next_valid]
        elif prev_valid is not None:
            result[i] = landmarks_list[prev_valid]
        elif next_valid is not None:
            result[i] = landmarks_list[next_valid]
    
    return result


def normalize_landmarks(landmarks: np.ndarray) -> np.ndarray:
    """Normaliza landmarks (centrar en muneca, escalar)."""
    normalized = np.copy(landmarks)
    
    for t in range(landmarks.shape[0]):
        frame = landmarks[t]
        wrist = frame[0]
        frame_centered = frame - wrist
        
        scale = np.linalg.norm(frame[12] - frame[0])
        if scale > 1e-6:
            frame_normalized = frame_centered / scale
        else:
            frame_normalized = frame_centered
        
        normalized[t] = frame_normalized
    
    return normalized


def process_video(video_id: str, hands_detector, max_frames: int = 64) -> Tuple[Optional[np.ndarray], Dict]:
    """Procesa un video completo."""
    stats = {
        'video_id': video_id,
        'success': False,
        'error': None,
        'sampled_frames': 0,
        'detected_frames': 0,
        'detection_rate': 0.0
    }
    
    video_path = RAW_VIDEOS_DIR / f"{video_id}.mp4"
    
    if not video_path.exists():
        stats['error'] = 'video_not_found'
        return None, stats
    
    frames = load_video_frames(video_path, max_frames)
    if len(frames) == 0:
        stats['error'] = 'no_frames_loaded'
        return None, stats
    
    stats['sampled_frames'] = len(frames)
    
    landmarks_list = []
    for frame in frames:
        lm = extract_landmarks_from_frame(frame, hands_detector)
        landmarks_list.append(lm)
        if lm is not None:
            stats['detected_frames'] += 1
    
    stats['detection_rate'] = stats['detected_frames'] / stats['sampled_frames']
    
    if stats['detection_rate'] < MIN_DETECTION_RATE:
        stats['error'] = f'low_detection_rate_{stats["detection_rate"]:.2f}'
        return None, stats
    
    landmarks_interpolated = interpolate_missing_frames(landmarks_list)
    landmarks_normalized = normalize_landmarks(landmarks_interpolated)
    
    stats['success'] = True
    return landmarks_normalized, stats


print("Funciones definidas.")

Funciones definidas.


In [7]:
# ==========================================
# 6. RECOPILAR VIDEO IDs
# ==========================================

def get_all_video_ids(split: str) -> List[Tuple[str, str, str]]:
    """Obtiene todos los video_ids de un split."""
    split_dir = WLASL100_ROOT / split
    video_ids = []
    
    for gloss_dir in split_dir.iterdir():
        if not gloss_dir.is_dir():
            continue
        gloss = gloss_dir.name
        
        for video_dir in gloss_dir.iterdir():
            if not video_dir.is_dir():
                continue
            video_id = video_dir.name
            video_ids.append((video_id, gloss, split))
    
    return video_ids

# Recopilar todos
all_videos = []
for split in ['train', 'val', 'test']:
    videos = get_all_video_ids(split)
    all_videos.extend(videos)
    print(f"{split}: {len(videos)} videos")

print(f"\nTotal: {len(all_videos)} videos a procesar")

train: 748 videos
val: 165 videos
test: 100 videos

Total: 1013 videos a procesar


In [8]:
# ==========================================
# 7. PROCESAR TODO WLASL100
# ==========================================

# Crear carpetas
for split in ['train', 'val', 'test']:
    (OUTPUT_ROOT / split).mkdir(parents=True, exist_ok=True)

print(f"Carpetas creadas en: {OUTPUT_ROOT}")

# Stats globales
global_stats = {
    'total': 0,
    'success': 0,
    'failed': 0,
    'errors': {},
    'detection_rates': [],
    'by_split': {'train': 0, 'val': 0, 'test': 0}
}

print(f"\nProcesando {len(all_videos)} videos...")
print("(Esto tardara unos 15-20 minutos)\n")

for video_id, gloss, split in tqdm(all_videos, desc="Procesando"):
    global_stats['total'] += 1
    
    landmarks, stats = process_video(video_id, detector, MAX_FRAMES)
    
    if stats['success']:
        output_dir = OUTPUT_ROOT / split / gloss
        output_dir.mkdir(parents=True, exist_ok=True)
        output_path = output_dir / f"{video_id}.npy"
        np.save(output_path, landmarks)
        
        global_stats['success'] += 1
        global_stats['by_split'][split] += 1
        global_stats['detection_rates'].append(stats['detection_rate'])
    else:
        global_stats['failed'] += 1
        error = stats['error']
        global_stats['errors'][error] = global_stats['errors'].get(error, 0) + 1

print("\n" + "="*50)
print("PROCESAMIENTO COMPLETADO")
print("="*50)

Carpetas creadas en: /home/fransando/Dev/tfg-sign-language-recognition/06_datasets/wlasl/landmarks/wlasl100

Procesando 1013 videos...
(Esto tardara unos 15-20 minutos)



Procesando: 100%|██████████| 1013/1013 [21:55<00:00,  1.30s/it]


PROCESAMIENTO COMPLETADO


In [9]:
# ==========================================
# 8. REPORTE
# ==========================================

print("=== REPORTE WLASL100 ===")
print(f"\nTotal: {global_stats['total']}")
print(f"  Exitosos: {global_stats['success']} ({100*global_stats['success']/global_stats['total']:.1f}%)")
print(f"  Fallidos: {global_stats['failed']} ({100*global_stats['failed']/global_stats['total']:.1f}%)")

print(f"\nPor split:")
for split, count in global_stats['by_split'].items():
    print(f"  {split}: {count}")

if global_stats['errors']:
    print(f"\nErrores:")
    for error, count in sorted(global_stats['errors'].items(), key=lambda x: -x[1]):
        print(f"  {error}: {count}")

if global_stats['detection_rates']:
    rates = np.array(global_stats['detection_rates'])
    print(f"\nDeteccion:")
    print(f"  Media: {rates.mean():.1%}")
    print(f"  Min: {rates.min():.1%}")
    print(f"  Max: {rates.max():.1%}")

=== REPORTE WLASL100 ===

Total: 1013
  Exitosos: 1002 (98.9%)
  Fallidos: 11 (1.1%)

Por split:
  train: 741
  val: 162
  test: 99

Errores:
  low_detection_rate_0.28: 4
  low_detection_rate_0.30: 3
  low_detection_rate_0.05: 1
  low_detection_rate_0.23: 1
  low_detection_rate_0.27: 1
  low_detection_rate_0.25: 1

Deteccion:
  Media: 68.3%
  Min: 31.2%
  Max: 100.0%


In [10]:
# ==========================================
# 9. GUARDAR METADATA
# ==========================================

landmarks_metadata = {
    'name': 'wlasl100_landmarks',
    'source': 'wlasl100',
    'description': 'MediaPipe Hand Landmarks from WLASL100',
    'landmark_shape': [21, 3],
    'max_frames': MAX_FRAMES,
    'min_detection_rate': MIN_DETECTION_RATE,
    'normalization': 'wrist_centered_scale_normalized',
    'interpolation': 'linear',
    'statistics': {
        'total_processed': global_stats['total'],
        'total_success': global_stats['success'],
        'total_failed': global_stats['failed'],
        'success_rate': global_stats['success'] / global_stats['total'] if global_stats['total'] > 0 else 0,
        'mean_detection_rate': float(np.mean(global_stats['detection_rates'])) if global_stats['detection_rates'] else 0,
        'by_split': global_stats['by_split']
    },
    'glosses': metadata['glosses'],
    'gloss_to_idx': metadata['gloss_to_idx'],
    'idx_to_gloss': metadata['idx_to_gloss'],
    'errors': global_stats['errors']
}

with open(OUTPUT_ROOT / 'metadata.json', 'w') as f:
    json.dump(landmarks_metadata, f, indent=2)

print(f"Metadata guardado: {OUTPUT_ROOT / 'metadata.json'}")

Metadata guardado: /home/fransando/Dev/tfg-sign-language-recognition/06_datasets/wlasl/landmarks/wlasl100/metadata.json


In [11]:
# ==========================================
# 10. VERIFICACION
# ==========================================

print("=== VERIFICACION ===")
for split in ['train', 'val', 'test']:
    npy_files = list((OUTPUT_ROOT / split).glob('**/*.npy'))
    if npy_files:
        sample = np.load(npy_files[0])
        print(f"{split}: {len(npy_files)} archivos, shape: {sample.shape}")

# Tamano
import subprocess
result = subprocess.run(['du', '-sh', str(OUTPUT_ROOT)], capture_output=True, text=True)
print(f"\nTamano total: {result.stdout.strip().split()[0]}")

=== VERIFICACION ===
train: 741 archivos, shape: (64, 21, 3)
val: 162 archivos, shape: (53, 21, 3)
test: 99 archivos, shape: (60, 21, 3)

Tamano total: 29M


In [12]:
# Liberar recursos
detector.close()
print("\nRecursos liberados.")
print(f"\nLandmarks WLASL100 guardados en: {OUTPUT_ROOT}")
print("Listo para entrenar con notebook 09!")


Recursos liberados.

Landmarks WLASL100 guardados en: /home/fransando/Dev/tfg-sign-language-recognition/06_datasets/wlasl/landmarks/wlasl100
Listo para entrenar con notebook 09!
